## Section 1 — Setup: Clone repo, install deps, GPU validation


In [ ]:
# Trở về thư mục gốc của Colab
%cd /content

# Xóa hoàn toàn folder project cũ (bao gồm cả data và models bên trong)
!rm -rf Sentiment-Analysis-Service

!git clone https://github.com/RocketUp-Team/Sentiment-Analysis-Service.git || true
%cd Sentiment-Analysis-Service
!git checkout feature/ai-core
!git pull


In [ ]:
!pip uninstall -y torchvision torchaudio

In [ ]:
!pip install -q -r requirements.txt


In [ ]:
!nvidia-smi

# Fail fast if GPU is not available — prevents silent CPU training
import torch
assert torch.cuda.is_available(), (
    '❌ CUDA not available! Go to Runtime > Change runtime type > GPU.'
)
print(f'✅ GPU ready: {torch.cuda.get_device_name(0)}')


## Section 2 — Credentials: Read Colab Secrets, pre-flight checks


In [ ]:
import os
from google.colab import userdata

os.environ['MLFLOW_TRACKING_URI'] = userdata.get('MLFLOW_TRACKING_URI')
os.environ['DAGSHUB_USER']        = userdata.get('DAGSHUB_USER')
os.environ['DAGSHUB_TOKEN']       = userdata.get('DAGSHUB_TOKEN')
os.environ['GITHUB_TOKEN']        = userdata.get('GITHUB_TOKEN')
os.environ['GIT_USER_EMAIL']      = userdata.get('GIT_USER_EMAIL')
os.environ['GIT_USER_NAME']       = userdata.get('GIT_USER_NAME')

# MODEL_VERSION is required — fail early with a clear message
_model_version = userdata.get('MODEL_VERSION')
if not _model_version or _model_version.strip().lower() in ('none', ''):
    raise ValueError(
        '❌ MODEL_VERSION secret is missing or empty. '
        'Add it in Colab Secrets (e.g. v1.2.0).'
    )
os.environ['MODEL_VERSION'] = _model_version.strip()

os.environ['MLFLOW_TRACKING_USERNAME'] = os.environ['DAGSHUB_USER']
os.environ['MLFLOW_TRACKING_PASSWORD'] = os.environ['DAGSHUB_TOKEN']

print(f'✅ Secrets loaded. MODEL_VERSION={os.environ["MODEL_VERSION"]}')


## Section 3 — Data Download


In [ ]:
!python -m src.data.downloader --task sarcasm
!python -m src.data.downloader --task sentiment


## Section 4 — Training


In [ ]:
import mlflow
from pathlib import Path
from src.scripts.run_finetuning import train
import os

model_version = os.environ.get('MODEL_VERSION', 'v1.0.0')

def _adapter_exists(task_name):
    return Path(f'models/adapters/{task_name}').exists()

tasks_to_train = [t for t in ['sarcasm', 'sentiment'] if not _adapter_exists(t)]

mlflow.set_experiment('colab_pipeline_run')

run_tags = {
    'model_version': model_version,
    'environment': 'colab',
    'dataset_version': 'v1',
}

with mlflow.start_run(run_name=f'full_pipeline_{model_version}') as parent_run:
    mlflow.set_tags(run_tags)
    if tasks_to_train:
        for task in tasks_to_train:
            with mlflow.start_run(run_name=f'train_{task}_{model_version}', nested=True):
                mlflow.set_tags(run_tags)
                train(task)
    else:
        print('⚠️  Both adapters already exist — skipping training.')
        mlflow.set_tag('training_skipped', 'true')

# Store run_id as a standalone variable — safe to use in all downstream cells
PARENT_RUN_ID = parent_run.info.run_id
print(f'✅ Training complete. PARENT_RUN_ID={PARENT_RUN_ID}')


## Section 5 — Evaluation


In [ ]:
from src.scripts.evaluate_finetuned import evaluate

with mlflow.start_run(run_id=PARENT_RUN_ID):
    metrics_sarcasm  = evaluate('sarcasm')
    mlflow.log_metric('sarcasm_overall_f1',  metrics_sarcasm['overall_f1'])

    metrics_sentiment = evaluate('sentiment')
    mlflow.log_metric('sentiment_overall_f1', metrics_sentiment['overall_f1'])

print('✅ Evaluation complete.')


## Section 6 — ONNX Export


In [ ]:
# 1. Export model Sentiment sang định dạng ONNX
!python -m src.scripts.export_onnx --adapter-name sentiment
# 2. Export model Sarcasm sang định dạng ONNX
!python -m src.scripts.export_onnx --adapter-name sarcasm
# 3. Chạy benchmark hiệu năng
!python -m src.scripts.benchmark_onnx

## Section 7 — Visualization


In [ ]:
# 1. Generate SHAP plots
!python -m src.scripts.generate_shap_plots --output-dir reports/shap_plots

# 2. Upload all reports to MLflow Artifacts under the parent run
import mlflow

try:
    with mlflow.start_run(run_id=PARENT_RUN_ID):
        mlflow.log_artifacts('reports', artifact_path='final_reports')
        print('✅ Reports logged to MLflow Artifacts!')
except Exception as exc:
    print(f'❌ Failed to log artifacts: {exc}')


## Section 8 — DVC Push + Git Push


In [ ]:
# --- DVC: authenticate and commit stage outputs ---
!dvc remote modify origin --local auth basic
!dvc remote modify origin --local user $DAGSHUB_USER
!dvc remote modify origin --local password $DAGSHUB_TOKEN
!dvc commit -f \
    download_sarcasm \
    download_sentiment \
    finetune_sarcasm \
    finetune_sentiment \
    export_onnx_sarcasm \
    export_onnx_sentiment \
    benchmark_onnx
!dvc push

# --- Git: use identity from Colab Secrets (not hardcoded) ---
!git config --global user.email "$GIT_USER_EMAIL"
!git config --global user.name  "$GIT_USER_NAME"

# Authenticate with GITHUB_TOKEN
!git remote set-url origin https://$GITHUB_TOKEN@github.com/RocketUp-Team/Sentiment-Analysis-Service.git

# Stage only non-DVC-tracked files:
#   dvc.lock  — DVC lock file (text, tracked by Git)
#   reports/  — JSON metrics and SHAP plots (not DVC outs)
# IMPORTANT: Do NOT git add models/onnx/ — it is a DVC out, managed by dvc push above
!git add dvc.lock reports/ || true
!git commit -m "chore: update dvc.lock for model-$MODEL_VERSION" || true
!git tag model-$MODEL_VERSION
!git push origin feature/ai-core
!git push origin model-$MODEL_VERSION
